# Análise de erva-mate: PCA

O conjunto de dados é composto por 30 amostras, resultantes de 6 pontos de tempo (0, 2, 4, 6, 9 e 12 meses) com 5 réplicas cada, avaliadas em 23 atributos físico-químicos, de cor, de compostos bioativos e sensoriais. Essa proporção entre número de amostras e número de variáveis (30 observações para 23 atributos, distribuídas em apenas 6 grupos temporais) é característica de estudos de alta dimensionalidade e baixo N, o que limita a robustez estatística de qualquer modelo ajustado sobre os dados e deve ser considerado na leitura de todos os resultados a seguir.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## Carregamento dos dados

Nesta primeira etapa, os dados são carregados a partir do Google Drive e é feita uma inspeção inicial do arquivo, conferindo as primeiras linhas, os nomes das colunas e o formato geral (número de linhas e colunas), antes de qualquer processamento.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(
    "/content/drive/MyDrive/dados-erva-mate/Dados FQ Bioativos Otavio.csv",
    sep=';',
    decimal=',',
    encoding='latin-1'
)

## Preparação dos dados

Antes de qualquer análise estatística, é preciso organizar os rótulos de tempo e garantir que todas as colunas de atributos estejam de fato em formato numérico. O rótulo original de tempo (T0, T2, T4...) é mantido para uso em gráficos, enquanto uma versão numérica é criada para os cálculos e para a ordenação correta das análises. Também é necessário tratar a conversão de vírgula para ponto decimal em colunas que eventualmente tenham sido lidas como texto.

In [ ]:
# --- Criar coluna numérica de tempo a partir do rótulo "T0", "T2", etc. ------
# Guardamos o rótulo original (para gráficos legíveis) e criamos uma versão
# numérica (para cálculos e ordenação correta no PLSR).
df["Tempo_label"] = df["Tempo"].astype(str)
df["Tempo_num"] = df["Tempo_label"].str.replace("T", "", case=False, regex=False).astype(float)

# Lista com o nome das colunas de atributos (todas menos identificadores de tempo/réplica)
colunas_atributos = [c for c in df.columns
                     if c not in ["Tempo", "Tempo_label", "Tempo_num", "Repetição"]]
print(f"\nNúmero de atributos encontrados: {len(colunas_atributos)}")
print(colunas_atributos)

In [ ]:
# Converter colunas de atributos para numéricas, tratando o separador decimal (vírgula).
# Mesmo com decimal=',' em pd.read_csv, algumas colunas podem ser interpretadas como 'object'
# (string) se houver inconsistências na formatação ou falha na inferência de tipo.
for col in colunas_atributos:
    if df[col].dtype == object:
        df[col] = (df[col].astype(str)
                   .str.replace('.', '', regex=False)
                   .str.replace(',', '.', regex=False)
                   .astype(float))
    else:
        df[col] = df[col].astype(float)

## 1. Análise de Componentes Principais (PCA)

A Análise de Componentes Principais foi aplicada aos 23 atributos padronizados (média 0, desvio padrão 1) com o objetivo de reduzir a dimensionalidade dos dados e identificar os principais eixos de variação entre as amostras, sem levar em conta a informação de tempo. Cada componente principal é uma combinação linear dos atributos originais, construída de forma a capturar o máximo de variância possível de forma sucessiva e ortogonal aos componentes anteriores.

In [ ]:

X = df[colunas_atributos].values
scaler = StandardScaler()
X_padronizado = scaler.fit_transform(X)

pca = PCA(n_components=2)
componentes = pca.fit_transform(X_padronizado)

# Guardamos os componentes de volta no dataframe para facilitar os gráficos
df["PC1"] = componentes[:, 0]
df["PC2"] = componentes[:, 1]

variancia_explicada = pca.explained_variance_ratio_
print(f"Variância explicada por PC1: {variancia_explicada[0]*100:.1f}%")
print(f"Variância explicada por PC2: {variancia_explicada[1]*100:.1f}%")
print(f"Total explicado pelos 2 componentes: {sum(variancia_explicada)*100:.1f}%")

In [ ]:
medias_por_tempo = df.groupby("Tempo_label")[colunas_atributos].mean()

# 2. Padroniza e ajusta o PCA usando APENAS as médias por tempo
scaler_medias = StandardScaler()
X_medias_padronizado = scaler_medias.fit_transform(medias_por_tempo)

# Queremos apenas as 2 primeiras componentes (PC1 e PC2)
pca_medias = PCA(n_components=2)
pca_medias.fit(X_medias_padronizado)

# Guarda a variância explicada por PC1 e PC2 para o título dos eixos
variancia_explicada = pca_medias.explained_variance_ratio_

# 3. Projeta os dados originais (réplicas) no espaço do PCA das médias
# (Para que as réplicas individuais fiquem na mesma escala e orientação das médias)
X_original_padronizado = scaler_medias.transform(df[colunas_atributos])
coordenadas_originais = pca_medias.transform(X_original_padronizado)

# Adiciona as coordenadas PC1 e PC2 de volta ao DataFrame original para plotagem
df["PC1"] = coordenadas_originais[:, 0]
df["PC2"] = coordenadas_originais[:, 1]

# 4. Projeta as médias por tempo no espaço do PCA
coordenadas_medias = pca_medias.transform(X_medias_padronizado)
medias_por_tempo_pca = pd.DataFrame(
    coordenadas_medias, columns=["PC1", "PC2"], index=medias_por_tempo.index
).reset_index()

# Mapeia ou traz de volta o "Tempo_num" para garantir a ordenação cronológica correta da linha
tempo_map = df[["Tempo_label", "Tempo_num"]].drop_duplicates()
medias_por_tempo_pca = medias_por_tempo_pca.merge(tempo_map, on="Tempo_label")
medias_por_tempo_pca = medias_por_tempo_pca.sort_values("Tempo_num")


# 5. Montagem do Gráfico exatamente como você pediu
ordem_tempos = df.sort_values("Tempo_num")["Tempo_label"].unique().tolist()

fig_nuvem = px.scatter(
    df,
    x="PC1",
    y="PC2",
    color="Tempo_label",
    category_orders={"Tempo_label": ordem_tempos},
    title="PCA - réplicas individuais por ponto de tempo (média)",
    opacity=0.75,
)

# Adiciona a trajetória central (média) usando o PCA calculado nas médias
fig_nuvem.add_trace(
    go.Scatter(
        x=medias_por_tempo_pca["PC1"],
        y=medias_por_tempo_pca["PC2"],
        mode="lines+markers+text",
        text=medias_por_tempo_pca["Tempo_label"],
        textposition="top center",
        line=dict(color="black", width=2, dash="dash"),
        marker=dict(size=10, color="black", symbol="diamond"),
        name="Média por tempo (trajetória)",
    )
)

# Configura os títulos dos eixos mostrando a variância de PC1 e PC2
fig_nuvem.update_layout(
    xaxis_title=f"PC1 ({variancia_explicada[0]*100:.1f}% da variância)",
    yaxis_title=f"PC2 ({variancia_explicada[1]*100:.1f}% da variância)",
)

fig_nuvem.show()

In [ ]:

ordem_tempos = df.sort_values("Tempo_num")["Tempo_label"].unique().tolist()

fig_nuvem = px.scatter(
    df, x="PC1", y="PC2",
    color="Tempo_label",
    category_orders={"Tempo_label": ordem_tempos},
    title="PCA - réplicas individuais por ponto de tempo",
    opacity=0.75
)

# Calcula a média de PC1/PC2 por tempo para desenhar a trajetória central
medias_por_tempo = (
    df.groupby(["Tempo_label", "Tempo_num"])[["PC1", "PC2"]]
    .mean()
    .reset_index()
    .sort_values("Tempo_num")
)

fig_nuvem.add_trace(go.Scatter(
    x=medias_por_tempo["PC1"], y=medias_por_tempo["PC2"],
    mode="lines+markers+text",
    text=medias_por_tempo["Tempo_label"],
    textposition="top center",
    line=dict(color="black", width=2, dash="dash"),
    marker=dict(size=10, color="black", symbol="diamond"),
    name="Média por tempo (trajetória)"
))

fig_nuvem.update_layout(
    xaxis_title=f"PC1 ({variancia_explicada[0]*100:.1f}% da variância)",
    yaxis_title=f"PC2 ({variancia_explicada[1]*100:.1f}% da variância)"
)
fig_nuvem.show()

### 1.1 Variância explicada por componente (scree plot)

O gráfico de scree apresenta a porcentagem de variância explicada por cada componente principal individualmente, o que ajuda a visualizar a partir de que ponto os componentes adicionais deixam de agregar informação relevante ao modelo.

In [ ]:

fig_scree = px.line(
    x=range(1, len(pca_completo.explained_variance_ratio_) + 1),
    y=pca_completo.explained_variance_ratio_ * 100,
    markers=True,
    labels={"x": "Componente Principal", "y": "% de Variância Explicada"},
    title="Scree Plot - Variância explicada por componente"
)
fig_scree.show()

### 1.2 Gráfico de escores (PC1 x PC2) e trajetória temporal

O gráfico de escores posiciona cada amostra no espaço definido pelos dois primeiros componentes principais, permitindo observar tanto a dispersão das réplicas dentro de cada tempo quanto a trajetória da média de cada tempo ao longo do eixo temporal, construída conectando os centroides de T0 a T12.

In [ ]:

ordem_tempos = df.sort_values("Tempo_num")["Tempo_label"].unique().tolist()

fig_nuvem = px.scatter(
    df, x="PC1", y="PC2",
    color="Tempo_label",
    category_orders={"Tempo_label": ordem_tempos},
    title="PCA - réplicas individuais por ponto de tempo",
    opacity=0.75
)

# Calcula a média de PC1/PC2 por tempo para desenhar a trajetória central
medias_por_tempo = (
    df.groupby(["Tempo_label", "Tempo_num"])[["PC1", "PC2"]]
    .mean()
    .reset_index()
    .sort_values("Tempo_num")
)

fig_nuvem.add_trace(go.Scatter(
    x=medias_por_tempo["PC1"], y=medias_por_tempo["PC2"],
    mode="lines+markers+text",
    text=medias_por_tempo["Tempo_label"],
    textposition="top center",
    line=dict(color="black", width=2, dash="dash"),
    marker=dict(size=10, color="black", symbol="diamond"),
    name="Média por tempo (trajetória)"
))

fig_nuvem.update_layout(
    xaxis_title=f"PC1 ({variancia_explicada[0]*100:.1f}% da variância)",
    yaxis_title=f"PC2 ({variancia_explicada[1]*100:.1f}% da variância)"
)
fig_nuvem.show()

### 1.3 Gráfico de *loadings*

O gráfico de *loadings* representa a contribuição (peso) de cada um dos 23 atributos originais na construção dos dois primeiros componentes principais. Vetores com maior magnitude indicam atributos que mais influenciam a posição das amostras no espaço de componentes, e vetores apontando em direções semelhantes indicam atributos correlacionados entre si nos dados.

In [ ]:

# Criamos o DataFrame de loadings usando o modelo ajustado com as médias (pca_medias)
loadings_medias = pd.DataFrame(
    pca_medias.components_.T,       # Transposto: uma linha por atributo, colunas para PC1 e PC2
    columns=["PC1", "PC2"],
    index=colunas_atributos
)

# Ordena os atributos pela importância absoluta na PC1 do modelo de médias
loadings_ordenados = loadings_medias.reindex(
    loadings_medias["PC1"].abs().sort_values(ascending=False).index
)

print("Atributos mais importantes para PC1 (Modelo de Médias):")
print(loadings_ordenados.head(10))

# Geração do Gráfico de Loadings baseado nas médias
fig_loadings = go.Figure()

for atributo in loadings_medias.index:
    fig_loadings.add_trace(go.Scatter(
        x=[0, loadings_medias.loc[atributo, "PC1"]],
        y=[0, loadings_medias.loc[atributo, "PC2"]],
        mode="lines+text",
        text=["", atributo],
        textposition="top center",
        showlegend=False
    ))

# Configurações de layout
fig_loadings.update_layout(
    title="Loadings do PCA (Baseado nas Médias) - contribuição de cada atributo",
    xaxis_title="PC1",
    yaxis_title="PC2",
    width=850,
    height=850,
    yaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=0.2
    )
)

fig_loadings.show()

In [ ]:

loadings = pd.DataFrame(
    pca.components_.T,          # transposto: uma linha por atributo
    columns=["PC1", "PC2"],
    index=colunas_atributos
)

loadings_ordenados = loadings.reindex(
    loadings["PC1"].abs().sort_values(ascending=False).index
)
print("Atributos mais importantes para PC1:")
print(loadings_ordenados.head(10))

# Gráfico de loadings (biplot simplificado)
fig_loadings = go.Figure()
for atributo in loadings.index:
    fig_loadings.add_trace(go.Scatter(
        x=[0, loadings.loc[atributo, "PC1"]],
        y=[0, loadings.loc[atributo, "PC2"]],
        mode="lines+text",
        text=["", atributo],
        textposition="top center",
        showlegend=False
    ))
fig_loadings.update_layout(
    title="Loadings do PCA - contribuição de cada atributo",
    xaxis_title="PC1", yaxis_title="PC2"
)

fig_loadings.update_layout(
    width=850,
    height=850,
    yaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=0.2
    )
)

fig_loadings.show()